In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

project_root = Path.cwd().parent
data_dir = project_root / 'data'
reports_dir = project_root / 'reports'
reports_dir.mkdir(exist_ok=True)
sys.path.insert(0, str(project_root / 'src'))

from train_model import RANDOM_STATE, load_and_encode, train_logistic_regression, train_xgboost

N_THRESHOLD_STEPS = 100
# Average quality loss stays <= 0 across the whole threshold range on this dataset, so the
# operating point is chosen against the worst single-query (max) loss instead.
MAX_QUALITY_LOSS_TARGET = 4.0  # no single routed-small query should lose more than 4 judge points

features_df = pd.read_parquet(data_dir / 'features.parquet')
dataset_df = pd.read_parquet(data_dir / 'query_dataset.parquet')[['query_id', 'small_model_score', 'large_model_score']]
merged = features_df.merge(dataset_df, on='query_id')
print(f"Loaded {len(merged)} rows")

In [ ]:
# Out-of-fold predicted probabilities -- each row's prediction comes from a model that
# never saw it during training, so the full 900-row set can be used for the Pareto curve
# without overstating performance the way training-set probabilities would.
def generate_oof_probabilities(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    lr_proba = np.zeros(len(X))
    xgb_proba = np.zeros(len(X))
    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr = y.iloc[train_idx]
        lr_model, scaler = train_logistic_regression(X_tr, y_tr)
        lr_proba[test_idx] = lr_model.predict_proba(scaler.transform(X_te))[:, 1]
        xgb_model = train_xgboost(X_tr, y_tr)
        xgb_proba[test_idx] = xgb_model.predict_proba(X_te)[:, 1]
    return pd.DataFrame({'lr_proba_small_ok': lr_proba, 'xgb_proba_small_ok': xgb_proba}, index=X.index)

X, y, _ = load_and_encode()
oof = generate_oof_probabilities(X, y)
merged = pd.concat([merged.reset_index(drop=True), oof.reset_index(drop=True)], axis=1)
quality_delta = (merged['large_model_score'] - merged['small_model_score']).to_numpy()
print(f"quality_delta (large - small): mean={quality_delta.mean():.3f}, max={quality_delta.max():.1f}")

In [ ]:
def sweep_thresholds(proba, quality_delta, n_steps=N_THRESHOLD_STEPS):
    rows = []
    for t in np.linspace(0, 1, n_steps + 1):
        routed_small = proba >= t
        pct_routed_small = routed_small.mean()
        deltas = quality_delta[routed_small]
        avg_quality_loss = deltas.mean() if routed_small.any() else 0.0
        p90_quality_loss = np.percentile(deltas, 90) if routed_small.any() else 0.0
        max_quality_loss = deltas.max() if routed_small.any() else 0.0
        rows.append({
            'threshold': round(float(t), 4), 'pct_routed_small': round(float(pct_routed_small), 4),
            'avg_quality_loss': round(float(avg_quality_loss), 4),
            'p90_quality_loss': round(float(p90_quality_loss), 4),
            'max_quality_loss': round(float(max_quality_loss), 4),
        })
    return pd.DataFrame(rows)

pareto_lr = sweep_thresholds(merged['lr_proba_small_ok'].to_numpy(), quality_delta)
pareto_xgb = sweep_thresholds(merged['xgb_proba_small_ok'].to_numpy(), quality_delta)
pareto_lr.to_csv(reports_dir / 'phase4_pareto_logistic_regression.csv', index=False)
pareto_xgb.to_csv(reports_dir / 'phase4_pareto_xgboost.csv', index=False)
pareto_lr.iloc[::10]

In [ ]:
# The headline chart: cost savings vs quality loss, both the average view (flat/negative
# across the whole curve -- a real finding, not a bug) and the tail-risk view (the metric
# that actually tells a decision-relevant story on this dataset).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(pareto_lr['pct_routed_small'], pareto_lr['avg_quality_loss'], marker='o', markersize=3, label='Logistic Regression')
ax1.plot(pareto_xgb['pct_routed_small'], pareto_xgb['avg_quality_loss'], marker='s', markersize=3, label='XGBoost')
ax1.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax1.set_xlabel('% of queries routed to small model (cost savings)')
ax1.set_ylabel('Avg quality loss on routed-small queries\n(large score - small score)')
ax1.set_title('Average quality loss (stays <= 0 across the whole curve)')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(pareto_lr['pct_routed_small'], pareto_lr['max_quality_loss'], marker='o', markersize=3, label='Logistic Regression')
ax2.plot(pareto_xgb['pct_routed_small'], pareto_xgb['max_quality_loss'], marker='s', markersize=3, label='XGBoost')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax2.set_xlabel('% of queries routed to small model (cost savings)')
ax2.set_ylabel('Worst single-query quality loss on routed-small queries')
ax2.set_title('Tail risk (max quality loss) -- the real decision curve')
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Cost Savings vs. Quality Loss Pareto Frontier')
fig.tight_layout()
fig.savefig(reports_dir / 'phase4_pareto_frontier.png', dpi=150)
plt.show()

In [ ]:
# Recommended operating threshold: maximize cost savings subject to a worst-case (max)
# per-query quality loss bound -- a real, business-defensible risk criterion, unlike an
# average-based target which is trivially satisfied everywhere on this dataset.
def recommend_threshold(pareto_df, max_quality_loss_target=MAX_QUALITY_LOSS_TARGET):
    ok = pareto_df[pareto_df['max_quality_loss'] <= max_quality_loss_target]
    if ok.empty:
        return {'threshold': 1.0, 'pct_routed_small': 0.0, 'avg_quality_loss': 0.0, 'max_quality_loss': 0.0, 'note': 'no threshold meets target'}
    best = ok.loc[ok['pct_routed_small'].idxmax()]
    return {
        'threshold': float(best['threshold']), 'pct_routed_small': float(best['pct_routed_small']),
        'avg_quality_loss': float(best['avg_quality_loss']), 'p90_quality_loss': float(best['p90_quality_loss']),
        'max_quality_loss': float(best['max_quality_loss']), 'max_quality_loss_target': max_quality_loss_target,
    }

rec_lr = recommend_threshold(pareto_lr)
rec_xgb = recommend_threshold(pareto_xgb)
recommendation = {
    'max_quality_loss_target': MAX_QUALITY_LOSS_TARGET,
    'logistic_regression': rec_lr,
    'xgboost': rec_xgb,
    'always_large_baseline': {'pct_routed_small': 0.0, 'avg_quality_loss': 0.0, 'max_quality_loss': 0.0},
    'always_small_baseline': {
        'pct_routed_small': 1.0,
        'avg_quality_loss': round(float(quality_delta.mean()), 4),
        'max_quality_loss': round(float(quality_delta.max()), 4),
    },
}
(reports_dir / 'phase4_threshold_recommendation.json').write_text(json.dumps(recommendation, indent=2))
print(json.dumps(recommendation, indent=2))